# GeoVision AI — Deployment & Case Study Analysis

Runs the trained ensemble (see `01_training_pipeline.ipynb`) on a real Azerbaijani study area
(Khirdalan-Masazir-Binagadi) across two dates, and produces the full multi-temporal urban-growth
analysis: land-cover segmentation, change detection, urban expansion quantification, development
hotspots, rule-based planning indicators, and reference-based consistency checks against
ESA WorldCover and NDWI (neither is ground truth).

**CPU-only** — no GPU required (inference on ~30 patches per date is fast even on CPU).

**Requires**, in `data/`:
- `2021_RGBNIR_aoi.npy`, `2026_RGBNIR_aoi.npy` — AOI-cropped Sentinel-2 imagery (see README for
  acquisition/preprocessing notes, including the BOA radiometric offset correction needed for
  post-2022-baseline products)
- Trained checkpoints (paths set in Part 1 below — see `checkpoints/README.md`)


## Part 1 — Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch, Rectangle

In [ ]:
class CFG:
    DATA_DIR = Path("data")
    FIG_DIR = Path("figures")

    DATE1_LABEL, DATE2_LABEL = "2021", "2026"
    DATE1_AOI_PATH = DATA_DIR / "2021_RGBNIR_aoi.npy"
    DATE2_AOI_PATH = DATA_DIR / "2026_RGBNIR_aoi.npy"
    DATE1_LC_PATH = DATA_DIR / "2021_landcover.npy"
    DATE2_LC_PATH = DATA_DIR / "2026_landcover.npy"

    TRANSITION_CSV_PATH = DATA_DIR / "transition_matrix.csv"
    EXPANSION_MASK_PATH = DATA_DIR / "nonbuiltup_to_builtup_mask.npy"
    HOTSPOT_GRID_PATH = DATA_DIR / "hotspot_grid.npy"

    # Update to your own checkpoint locations -- see checkpoints/README.md
    ENSEMBLE_CONFIGS = [
        {"path": "checkpoints/unetpp_best_model.pth", "model_name": "UnetPlusPlus", "encoder": "efficientnet-b5"},
        {"path": "checkpoints/unet_best_model.pth",    "model_name": "Unet",         "encoder": "efficientnet-b3"},
    ]

    IN_CHANNELS = 5
    NUM_CLASSES = 7
    IGNORE_INDEX = 0
    PATCH_SIZE = 256
    BATCH_SIZE = 16
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    PIXELS_PER_KM2 = 10_000
    CELL_SIZE_PX = 50   # hotspot grid cell size (500m at 10m/px)

    AOI_BOUNDS = (49.73, 40.41, 49.87, 40.53)   # min_lon, min_lat, max_lon, max_lat
    AOI_NAME = "Khirdalan - Masazir - Binagadi, Azerbaijan"

    CLASS_NAMES = ["Ignore", "Tree cover", "Low vegetation", "Cropland",
                   "Built-up", "Bare/sparse", "Water/wetland"]
    CLASS_COLORS = ["#FFFFFF", "#1A6B1A", "#8FCB6E", "#E8C24A",
                     "#C0392B", "#B08D57", "#3B82C4"]
    GREEN_IDS = (1, 2)
    BUILTUP_ID = 4

    WORLDCOVER_TIF_PATH = DATA_DIR / "worldcover2021_aoi_remapped.npy"
    

CFG.FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", CFG.DEVICE)

## Part 2 — Segmentation 

Pads each AOI crop to a multiple of 256px, patchifies, appends the NDVI channel (matching
training), runs the ensemble (softmax-averaged), and stitches predictions back into a full-
resolution land-cover map per date.

In [ ]:
def add_ndvi_channel(img_uint8):
    img = img_uint8.astype(np.float32)
    red, nir = img[..., 0], img[..., 3]
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndvi_scaled = ((ndvi + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    return np.dstack([img_uint8, ndvi_scaled])


def pad_to_multiple(arr, patch_size=CFG.PATCH_SIZE):
    h, w = arr.shape[:2]
    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size
    padded = np.pad(arr, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant", constant_values=0)
    return padded, (h, w)


def patchify(padded_arr, patch_size=CFG.PATCH_SIZE):
    h, w = padded_arr.shape[:2]
    patches, coords = [], []
    for y in range(0, h, patch_size):
        for x in range(0, w, patch_size):
            patches.append(padded_arr[y:y + patch_size, x:x + patch_size])
            coords.append((y, x))
    return patches, coords


def load_ensemble_model(cfg, device):
    ModelClass = getattr(smp, cfg["model_name"])
    model = ModelClass(
        encoder_name=cfg["encoder"], encoder_weights=None,
        in_channels=CFG.IN_CHANNELS, classes=CFG.NUM_CLASSES, activation=None,
    ).to(device)
    ckpt = torch.load(cfg["path"], map_location=device, weights_only=False)
    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    print(f"Loaded {cfg['model_name']}/{cfg['encoder']} from {cfg['path']}")
    return model


def run_inference_ensemble(models, patches, coords, padded_shape, device, batch_size=CFG.BATCH_SIZE):
    full_pred = np.zeros(padded_shape[:2], dtype=np.uint8)
    with torch.no_grad():
        for i in range(0, len(patches), batch_size):
            batch_patches = patches[i:i + batch_size]
            batch_coords = coords[i:i + batch_size]
            batch_arr = np.stack([add_ndvi_channel(p) for p in batch_patches]).astype(np.float32) / 255.0
            batch_tensor = torch.from_numpy(batch_arr.transpose(0, 3, 1, 2)).to(device)

            probs_sum = None
            for m in models:
                probs = torch.softmax(m(batch_tensor), dim=1)
                probs_sum = probs if probs_sum is None else probs_sum + probs
            preds = torch.argmax(probs_sum / len(models), dim=1).cpu().numpy()

            for pred, (y, x) in zip(preds, batch_coords):
                full_pred[y:y + CFG.PATCH_SIZE, x:x + CFG.PATCH_SIZE] = pred
    return full_pred


def segment_full_area(models, rgbnir_uint8, device):
    padded, orig_shape = pad_to_multiple(rgbnir_uint8)
    patches, coords = patchify(padded)
    print(f"  {len(patches)} patches ({padded.shape[:2]} padded from {orig_shape})")
    full_pred_padded = run_inference_ensemble(models, patches, coords, padded.shape, device)
    return full_pred_padded[:orig_shape[0], :orig_shape[1]]

In [ ]:
ensemble_models = [load_ensemble_model(cfg, CFG.DEVICE) for cfg in CFG.ENSEMBLE_CONFIGS]

date1_crop = np.load(CFG.DATE1_AOI_PATH)
date2_crop = np.load(CFG.DATE2_AOI_PATH)
print(f"{CFG.DATE1_LABEL} crop shape:", date1_crop.shape)
print(f"{CFG.DATE2_LABEL} crop shape:", date2_crop.shape)

print(f"\nSegmenting {CFG.DATE1_LABEL}...")
date1_landcover = segment_full_area(ensemble_models, date1_crop, CFG.DEVICE)
print(f"Segmenting {CFG.DATE2_LABEL}...")
date2_landcover = segment_full_area(ensemble_models, date2_crop, CFG.DEVICE)

np.save(CFG.DATE1_LC_PATH, date1_landcover)
np.save(CFG.DATE2_LC_PATH, date2_landcover)
print("Saved land-cover maps.")

## Part 3 — Land-use statistics 

In [ ]:
def compute_area_stats(landcover):
    counts = np.bincount(landcover.flatten(), minlength=CFG.NUM_CLASSES)
    valid = counts[1:].sum()
    stats = {}
    for i in range(1, CFG.NUM_CLASSES):
        stats[CFG.CLASS_NAMES[i]] = {
            "km2": counts[i] / CFG.PIXELS_PER_KM2,
            "pct": (counts[i] / valid * 100) if valid > 0 else 0.0,
        }
    green_px = sum(counts[c] for c in CFG.GREEN_IDS)
    stats["Green space"] = {
        "km2": green_px / CFG.PIXELS_PER_KM2,
        "pct": (green_px / valid * 100) if valid > 0 else 0.0,
    }
    return stats


stats1 = compute_area_stats(date1_landcover)
stats2 = compute_area_stats(date2_landcover)

for label, stats in [(CFG.DATE1_LABEL, stats1), (CFG.DATE2_LABEL, stats2)]:
    print(f"\n=== {label} ===")
    for name, v in stats.items():
        print(f"  {name:16s}: {v['km2']:7.3f} km^2  ({v['pct']:5.2f}%)")

## Part 4 — Change detection 

In [ ]:
def build_transition_matrix(date1, date2):
    valid = (date1 != CFG.IGNORE_INDEX) & (date2 != CFG.IGNORE_INDEX)
    d1, d2 = date1[valid], date2[valid]
    matrix = np.zeros((CFG.NUM_CLASSES, CFG.NUM_CLASSES), dtype=np.int64)
    for i in range(CFG.NUM_CLASSES):
        for j in range(CFG.NUM_CLASSES):
            matrix[i, j] = np.sum((d1 == i) & (d2 == j))
    return matrix


def compute_nonbuiltup_to_builtup(date1, date2):
    was_builtup = date1 == CFG.BUILTUP_ID
    now_builtup = date2 == CFG.BUILTUP_ID
    was_ignore = (date1 == CFG.IGNORE_INDEX) | (date2 == CFG.IGNORE_INDEX)
    return now_builtup & ~was_builtup & ~was_ignore


matrix = build_transition_matrix(date1_landcover, date2_landcover)
matrix_df = pd.DataFrame(matrix, index=CFG.CLASS_NAMES, columns=CFG.CLASS_NAMES)
matrix_df.to_csv(CFG.TRANSITION_CSV_PATH)

expansion_mask = compute_nonbuiltup_to_builtup(date1_landcover, date2_landcover)
np.save(CFG.EXPANSION_MASK_PATH, expansion_mask)

total_expansion_km2 = expansion_mask.sum() / CFG.PIXELS_PER_KM2
print(f"Total non-built-up -> built-up: {total_expansion_km2:.3f} km^2")
print("\nBy source class:")
for src in CFG.CLASS_NAMES:
    if src in ("Ignore", "Built-up"):
        continue
    px = matrix_df.loc[src, "Built-up"]
    if px > 0:
        print(f"  {src:16s} -> Built-up: {px / CFG.PIXELS_PER_KM2:7.3f} km^2")

builtup1 = matrix_df.loc["Built-up"].sum() / CFG.PIXELS_PER_KM2
builtup2 = matrix_df["Built-up"].sum() / CFG.PIXELS_PER_KM2
print(f"\nBuilt-up {CFG.DATE1_LABEL}: {builtup1:.3f} km^2 | {CFG.DATE2_LABEL}: {builtup2:.3f} km^2 "
      f"| Growth: {builtup2-builtup1:+.3f} km^2 ({(builtup2-builtup1)/builtup1*100:+.1f}%)")

## Part 5 — Development hotspots 

In [ ]:
def build_hotspot_grid(expansion_mask, cell_size=CFG.CELL_SIZE_PX):
    h, w = expansion_mask.shape
    pad_h = (cell_size - h % cell_size) % cell_size
    pad_w = (cell_size - w % cell_size) % cell_size
    padded = np.pad(expansion_mask, ((0, pad_h), (0, pad_w)), mode="constant", constant_values=False)
    grid_h, grid_w = padded.shape[0] // cell_size, padded.shape[1] // cell_size
    return padded.reshape(grid_h, cell_size, grid_w, cell_size).sum(axis=(1, 3)).astype(np.int64)


hotspot_grid = build_hotspot_grid(expansion_mask)
np.save(CFG.HOTSPOT_GRID_PATH, hotspot_grid)

flat_idx = np.argsort(hotspot_grid.flatten())[::-1][:10]
rows, cols = np.unravel_index(flat_idx, hotspot_grid.shape)
print("Top 10 hotspot cells (500m x 500m):")
for r, c in zip(rows, cols):
    px = hotspot_grid[r, c]
    if px == 0:
        continue
    print(f"  rows {r*CFG.CELL_SIZE_PX}-{(r+1)*CFG.CELL_SIZE_PX}, "
          f"cols {c*CFG.CELL_SIZE_PX}-{(c+1)*CFG.CELL_SIZE_PX}: "
          f"{px/CFG.PIXELS_PER_KM2:.4f} km^2")

## Part 6 — Planning indicators 

In [ ]:
def compute_planning_indicators(stats1, stats2, expansion_mask, hotspot_grid):
    indicators = []

    green_pct = stats2["Green space"]["pct"]
    status = "OK" if green_pct >= 15.0 else "FLAG"
    indicators.append(("Green space ratio", status, f"{green_pct:.1f}%"))

    crop1, crop2 = stats1["Cropland"]["km2"], stats2["Cropland"]["km2"]
    loss_pct = (crop1 - crop2) / crop1 * 100 if crop1 > 0 else 0.0
    status = "FLAG" if loss_pct >= 25.0 else "WATCH"
    indicators.append(("Agricultural land loss", status, f"{loss_pct:.0f}%"))

    builtup2 = stats2["Built-up"]["pct"]
    status = "FLAG" if builtup2 >= 60 else ("WATCH" if builtup2 >= 55 else "OK")
    indicators.append(("Built-up saturation", status, f"{builtup2:.1f}%"))

    total_km2 = expansion_mask.sum() / CFG.PIXELS_PER_KM2
    top10_km2 = np.sort(hotspot_grid.flatten())[::-1][:10].sum() / CFG.PIXELS_PER_KM2
    diffuse_pct = (total_km2 - top10_km2) / total_km2 * 100 if total_km2 > 0 else 0
    status = "FLAG" if diffuse_pct >= 70 else "OK"
    indicators.append(("Development pattern", status, f"{diffuse_pct:.0f}% diffuse"))

    return indicators


for name, status, value in compute_planning_indicators(stats1, stats2, expansion_mask, hotspot_grid):
    print(f"[{status:5s}] {name}: {value}")

## Part 7 — Consistency checks against reference data

No pixel-level ground truth exists for this AOI, so two reference-based consistency checks are
used. Neither is ground truth:
1. **ESA WorldCover 2021** — the source of the training labels (a model-derived product with its
   own errors). The AOI lies outside every training raster, so this is a held-out-area check, but
   it measures consistency with a reference product, not accuracy. Note: grid-resized
   (nearest-neighbor) to match the model's array shape rather than fully CRS-reprojected — an
   approximate check, not survey-grade. Reports overall pixel agreement and per-class IoU.
2. **NDWI** — a physics-based water index requiring no training data, used as a reference water
   mask for the Water/wetland class. Shadows (including cloud shadow) can also have NDWI > 0.
   Reports precision, recall and IoU.

In [ ]:
if CFG.WORLDCOVER_TIF_PATH.exists():
    wc_lc = np.load(CFG.WORLDCOVER_TIF_PATH)
    valid = (date1_landcover != 0) & (wc_lc != 0)
    agreement = (date1_landcover == wc_lc)[valid].sum() / valid.sum() * 100
    print(f"WorldCover overall pixel agreement ({CFG.DATE1_LABEL}): {agreement:.1f}%")

    pred, ref = date1_landcover[valid], wc_lc[valid]
    print("Per-class IoU vs WorldCover (%):")
    for cid in range(1, CFG.NUM_CLASSES):
        tp = ((pred == cid) & (ref == cid)).sum()
        union = ((pred == cid) | (ref == cid)).sum()
        iou = tp / union * 100 if union > 0 else float("nan")
        print(f"  {CFG.CLASS_NAMES[cid]:16s} {iou:5.1f}")
else:
    print("WorldCover reference not found -- skipping this check.")

In [ ]:
def compute_ndwi(rgbnir_uint8):
    img = rgbnir_uint8.astype(np.float32)
    green, nir = img[..., 1], img[..., 3]
    return (green - nir) / (green + nir + 1e-6)


for label, rgbnir, lc in [(CFG.DATE1_LABEL, date1_crop, date1_landcover),
                            (CFG.DATE2_LABEL, date2_crop, date2_landcover)]:
    ndwi_water = compute_ndwi(rgbnir) > 0.0
    model_water = lc == 6
    tp = (model_water & ndwi_water).sum()
    precision = tp / model_water.sum() if model_water.sum() > 0 else float("nan")
    recall = tp / ndwi_water.sum() if ndwi_water.sum() > 0 else float("nan")
    iou = tp / (model_water | ndwi_water).sum()
    print(f"{label}: model water {model_water.sum()/CFG.PIXELS_PER_KM2:.2f} km^2, "
          f"NDWI water {ndwi_water.sum()/CFG.PIXELS_PER_KM2:.2f} km^2 | "
          f"precision {precision:.2f}, recall {recall:.2f}, IoU {iou:.2f}")

## Part 8 — Figures 

Produces the report figure set: study area, segmentation maps, class highlights, side-by-side
comparison, change detection, urban expansion, and development hotspots.

In [ ]:
def landcover_cmap_norm():
    cmap = ListedColormap(CFG.CLASS_COLORS)
    norm = BoundaryNorm(list(range(len(CFG.CLASS_COLORS) + 1)), cmap.N)
    return cmap, norm


def save_fig(fig, name):
    path = CFG.FIG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved", path)


def highlight_fig(landcover, class_ids, color, name, year):
    """One class group highlighted on a light-grey background (figures 3 and 4)."""
    mask = np.isin(landcover, class_ids)
    km2 = mask.sum() / CFG.PIXELS_PER_KM2
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(np.zeros(landcover.shape), cmap=ListedColormap(["#E6E6E6"]), vmin=0, vmax=1)
    ax.imshow(np.ma.masked_where(~mask, mask), cmap=ListedColormap([color]), vmin=0, vmax=1)
    ax.set_title(f"{name} -- {year} ({mask.mean() * 100:.1f}% of area, {km2:.1f} km^2)")
    ax.axis("off")
    return fig


def change_detection_fig(lc1, lc2):
    """Per-pixel change categories between the two dates (figure 6)."""
    valid = (lc1 != CFG.IGNORE_INDEX) & (lc2 != CFG.IGNORE_INDEX)
    changed = (lc1 != lc2) & valid
    veg_to_built = np.isin(lc1, CFG.GREEN_IDS) & (lc2 == CFG.BUILTUP_ID)
    crop_to_built = (lc1 == 3) & (lc2 == CFG.BUILTUP_ID)
    bare_to_built = (lc1 == 5) & (lc2 == CFG.BUILTUP_ID)
    other_change = changed & ~veg_to_built & ~crop_to_built & ~bare_to_built

    change_map = np.zeros(lc1.shape, dtype=np.uint8)
    change_map[other_change] = 4
    change_map[bare_to_built] = 3
    change_map[crop_to_built] = 2
    change_map[veg_to_built] = 1

    colors = ["#D9D9D9", "#F39C12", "#C0392B", "#8E5A2B", "#95A5A6"]
    names = ["Unchanged", "Vegetation -> Built-up", "Cropland -> Built-up",
             "Bare -> Built-up", "Other change (unverified; may include noise)"]
    areas = [(valid & ~changed).sum(), veg_to_built.sum(), crop_to_built.sum(),
             bare_to_built.sum(), other_change.sum()]

    fig, ax = plt.subplots(figsize=(9, 7))
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], 5)
    ax.imshow(np.ma.masked_where(~valid, change_map), cmap=ListedColormap(colors), norm=norm)
    ax.set_title(f"Change Detection: {CFG.DATE1_LABEL} -> {CFG.DATE2_LABEL}")
    ax.axis("off")
    ax.legend(handles=[Patch(facecolor=c, label=f"{n} ({a / CFG.PIXELS_PER_KM2:.2f} km^2)")
                       for c, n, a in zip(colors, names, areas)],
              loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
    return fig


# Figure 1 -- study area
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(date1_crop[..., :3])
ax.set_title(f"Study Area -- {CFG.AOI_NAME}")
save_fig(fig, "fig1_study_area.png")

# Figure 2 -- segmentation
cmap, norm = landcover_cmap_norm()
legend_handles = [Patch(facecolor=CFG.CLASS_COLORS[i], label=CFG.CLASS_NAMES[i]) for i in range(1, 7)]
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(date2_landcover, cmap=cmap, norm=norm)
ax.set_title(f"Land-Cover Segmentation -- {CFG.DATE2_LABEL}")
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
save_fig(fig, "fig2_segmentation.png")

# Figures 3 and 4 -- built-up and vegetation, both dates
for label, lc in [(CFG.DATE1_LABEL, date1_landcover), (CFG.DATE2_LABEL, date2_landcover)]:
    save_fig(highlight_fig(lc, [CFG.BUILTUP_ID], "#C0392B", "Built-up", label),
             f"fig3_builtup_{label}.png")
for label, lc in [(CFG.DATE1_LABEL, date1_landcover), (CFG.DATE2_LABEL, date2_landcover)]:
    save_fig(highlight_fig(lc, CFG.GREEN_IDS, "#2E8B2E", "Vegetation (tree cover + low vegetation)", label),
             f"fig4_vegetation_{label}.png")

# Figure 5 -- side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
axes[0].imshow(date1_landcover, cmap=cmap, norm=norm)
axes[0].set_title(CFG.DATE1_LABEL)
axes[1].imshow(date2_landcover, cmap=cmap, norm=norm)
axes[1].set_title(CFG.DATE2_LABEL)
fig.legend(handles=legend_handles, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.05), fontsize=9)
fig.suptitle("Land-Cover Comparison", fontsize=14)
save_fig(fig, "fig5_comparison.png")

# Figure 6 -- change detection
save_fig(change_detection_fig(date1_landcover, date2_landcover), "fig6_change_detection.png")

# Figure 7 -- urban expansion
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(date2_crop[..., :3])
ax.imshow(np.ma.masked_where(~expansion_mask, expansion_mask), cmap=ListedColormap(["#FF3B30"]), alpha=0.75)
ax.set_title(f"Urban Expansion {CFG.DATE1_LABEL}-{CFG.DATE2_LABEL} "
             f"({expansion_mask.sum()/CFG.PIXELS_PER_KM2:.2f} km^2)")
save_fig(fig, "fig7_expansion_map.png")

# Figure 8 -- hotspots
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(date2_crop[..., :3])
max_count = hotspot_grid.max() if hotspot_grid.max() > 0 else 1
for r in range(hotspot_grid.shape[0]):
    for c in range(hotspot_grid.shape[1]):
        val = hotspot_grid[r, c]
        if val == 0:
            continue
        intensity = float(np.clip(val / max_count, 0.1, 0.9))
        y0, x0 = r * CFG.CELL_SIZE_PX, c * CFG.CELL_SIZE_PX
        ax.add_patch(Rectangle((x0, y0), CFG.CELL_SIZE_PX, CFG.CELL_SIZE_PX,
                                facecolor="#FF3B30", alpha=intensity, edgecolor="none"))
ax.set_title("Urban Development Hotspots")
save_fig(fig, "fig8_hotspots.png")

print("\nAll figures saved to", CFG.FIG_DIR.resolve())

## Next step

Launch the interactive dashboard (`app.py`, in the repo root) to explore these results:

```bash
streamlit run app.py
```